In [ ]:
# SIFT connected code

In [ ]:
# The SIFT calculations were performed by SIFT 4G avaible here: https://sift-dna.org/sift4g # Momentálne nefunguje

In [ ]:
# After SIFT calculations of protein pathogenicity, we processed results subsequently:
# For every protein we dowloanded matrix directly from the web and saved it under /SIFT_protein name.txt/ which can be found in ../GLUT project documentation/Results from SIFT/
data = []

with open("../SIFT/SIFT_GLUT14.txt", 'r') as f:
    for line in f:
        parts = line.strip().split()

        # Skips headers or rows with non-numeric values
        if not parts or not parts[0][0].isdigit():
            continue

        pos_amk = parts[0]
        try:
            dk = float(parts[1])
            values = list(map(float, parts[2:]))
        except ValueError:
            continue  # skips if any value is not a number

        # Extracting position and AMK
        import re
        match = re.match(r"(\d+)([A-Z])", pos_amk)
        if match:
            pos, amk = match.groups()

            # (sum(values) - 1) / 19
            if len(values) == 20:  # sanity check
                ave = (sum(values) - 1) / 19
            else:
                ave = None  # unexpected format

            data.append([pos, amk, dk, ave])

# Creating a DataFrame and saving it
import pandas as pd
df = pd.DataFrame(data, columns=['pos', 'AMK', 'dk', 'ave'])
df.to_csv('SIFTcounted_average protein name.txt', sep='\t', index=False)

# This part is written just for a single protein, so you have to reapeat it 14 times. 
# Thorugh this part we counted averages for each position of aminoacid and saved it under the protein name. Can be found in ../GLUT project documentation/Results from SIFT/Averages/

In [ ]:
# Subsequent calculation of the pathogenicity average for the entire protein. 
# Again, this step must be repeated for each protein separately, i.e. 14 times.
import os
import pandas as pd

folder_path = "../GLUT project documentation/Results from SIFT/Averages"  

# Final list for saving (file name, diameter)
results = []

# Browsing all files in a folder
for filename in os.listdir(folder_path):
    if filename.endswith('.txt'):  
        file_path = os.path.join(folder_path, filename)
        
        # Loading the data
        df = pd.read_csv(file_path, sep='\t')
        
        # Calculating the average from the 'ave' column
        avg = df['ave'].mean()
        
        # Add to results (file name and average)
        results.append([filename, avg])

# Creating a DataFrame from the results
result_df = pd.DataFrame(results, columns=['file', 'average'])

# USave to a new text file
result_df.to_csv('pathogenicity_SIFT.txt', sep='\t', index=False)

# In each file will be newly counted average of the pathogenicity for the protein. 
# These numbers we rewrite to the .txt file named /pathogenicity_SIFT.txt/

In [ ]:
# In the next step, to determine the pathogenicity of the protein (intracellular/extracellular domain and transmembrane part), we will use the sorting from the "AlphaMissense connected code GLUT" code.
# We will use the folder with the saved .xlsx files as follows:
import pandas as pd

# Loading the main file
df = pd.read_excel("../GLUT project documentation/Results from DeepTMHMM/Excel output from sequences/protein name_AMK_output_MIO.xlsx")
df.columns = [col.strip() for col in df.columns]

# Converting positions to whole numbers
for col in ['O_position', 'M_position', 'I_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Loading SIFT results with header (pos, AMK, dk, ave)
df_scores = pd.read_csv("../GLUT project documentation/Results from SIFT/Averages/SIFTcounted_average protein name.txt", sep='\t')

# The position is already in the 'pos' column as a number (check that it is int)
df_scores['pos'] = pd.to_numeric(df_scores['pos'], errors='coerce').astype('Int64')

# We will create a map pos -> ave
score_map = df_scores.set_index('pos')['ave'].to_dict()

# Mapping scores from 'ave' to individual positions in df
df['O_patogenicity'] = df['O_position'].map(score_map)
df['M_patogenicity'] = df['M_position'].map(score_map)
df['I_patogenicity'] = df['I_position'].map(score_map)

# Output
print(df.head())

# Saving the output to the excel file
df.to_excel("SIFToutputMIO_protein name.xlsx", index=False)
# Again, this step have to be done for each protein separately

In [ ]:
# The pathogenicity averages for individual protein parts were calculated in a newly created .xlsx files: /output_with_pathogenicity_PolyPhen-2MIO_"protein name".xlsx/ using the AVERAGE function. 
# The calculated average was copied to a .txt file using the following code:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "../GLUT project documentation/Results from DeepTMHMM/SIFT"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("O_patogenicity", None),
                row.get("M_patogenicity", None),
                row.get("I_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "O_patogenicity", "M_patogenicity", "I_patogenicity"])

# Save to text file
result_df.to_csv("SIFT(MIO).txt", sep="\t", index=False)

# Showing the output
print(result_df.head())

In [ ]:
# We will use a similar procedure for lining residues, binding places, and lining residues without binding places, which were calculated and stored using the "AlphaMissense connected code" code.

In [ ]:
import pandas as pd

# Loading the main file
df = pd.read_excel("../GLUT project documentation/Binding places and lining residues/Excel files/protein name analysis lr_bp_lr-bp.xlsx")
df.columns = [col.strip() for col in df.columns]

# Converting positions to whole numbers
for col in ['lr_position', 'bp_position', 'lr-bp_position']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

# Loading SIFT results with header (pos, AMK, dk, ave)
df_scores = pd.read_csv("../GLUT project documentation/Results from SIFT/Averages/SIFTcounted_average protein name.txt", sep='\t')

# The position is already in the 'pos' column as a number (check that it is int)
df_scores['pos'] = pd.to_numeric(df_scores['pos'], errors='coerce').astype('Int64')

# We will create a map pos -> ave
score_map = df_scores.set_index('pos')['ave'].to_dict()

# Mapping scores from 'ave' to individual positions in df
df['lr_patogenicity'] = df['lr_position'].map(score_map)
df['bp_patogenicity'] = df['bp_position'].map(score_map)
df['lr-bp_patogenicity'] = df['lr-bp_position'].map(score_map)

# Output
print(df.head())

# Saving the output to the excel file
df.to_excel("SIFToutputlr,bp,lr-bp_protein name.xlsx", index=False)
# Again, this step have to be done for each protein separately

In [ ]:
import pandas as pd
import os

# Set the path to the folder with the files
folder_path = "../GLUT project documentation/Binding places and lining residues/SIFT"  # A folder containing all newly created files with calculated averages

# Initialization of the list for results
results = []

# Going through all files in the folder
for file in os.listdir(folder_path):
    if file.endswith(".xlsx"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_excel(file_path)
        
        # Find a row with the value 'AVERAGE' in any column
        avg_row = df[df.apply(lambda row: row.astype(str).str.contains("AVERAGE", case=False).any(), axis=1)]
        
        if not avg_row.empty:
            row = avg_row.iloc[0]
            values = [
                row.get("lr_patogenicity", None),
                row.get("bp_patogenicity", None),
                row.get("lr-bp_patogenicity", None)
            ]
            results.append([file, *values])

# Conversion to DataFrame
result_df = pd.DataFrame(results, columns=["filename", "lr_patogenicity", "bp_patogenicity", "lr-bp_patogenicity"])

# Save to text file
result_df.to_csv("SIFT(lr,bp,lr-bp).txt", sep="\t", index=False)

# Showing the output
print(result_df.head())


In [ ]:
# Merging all three newly created .txt files together
import pandas as pd
# Loading three files
df1 = pd.read_csv("../GLUT project documentation/Results from SIFT/pathogenicity_SIFT.txt", sep="\t") 
df2 = pd.read_csv("../GLUT project documentation/Results from DeepTMHMM/SIFT/SIFT(MIO).txt", sep="\t")
df3 = pd.read_csv("../GLUT project documentation/Binding places and lining residues/SIFT/SIFT(lr,bp,lr-bp).txt", sep="\t")

# Merge data according to the common column 'filename'. This column contains the names of proteins.
merged_df = df1.merge(df2, on="filename", how="outer")
merged_df = merged_df.merge(df3, on="filename", how="outer")

# Saving the resulting file
merged_df.to_csv("connected_file_SIFT.txt", sep="\t", index=False)


In [ ]:
#Creating the heatmap
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Loading the file
df = pd.read_csv("../GLUT project documentation/Conected results/connected_file_SIFT.txt", sep="\t")

# Setting 'filename' as index
df.set_index("protein", inplace=True)

# Creating a heatmap 
plt.figure(figsize=(10, len(df) * 0.4))
sns.heatmap(
    df,
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    linecolor='gray',
    annot=True,
    fmt=".3f"  # Formátovanie na 2 desatinné miesta
)


plt.title("GLUTs pathogenicity profile SIFT", fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()


plt.savefig("heatmap_patogenicitySIFT.png", dpi=300, bbox_inches='tight')

plt.show()